# Step 1.4c: Three open questions about the SpeechArenaBench labels

**Run on the server** (kernel = the `hinglishtts` venv). **Read-only.** It reads the cached table from the previous notebook (`analysis/step1_4b/sab_meta.parquet`, our 4,444 allowed-system pairs; no audio) and saves results under `analysis/step1_4c/`.

Each row of the data is **one rater's vote** on a pair. What we still need to know before writing the loader:

1. **What does a value like "Gemini 2.5 Pro TTS, Sonic 3" mean** (both good, or both bad)? We check it against the six axis ratings (1 to 5) the same rater gave to each clip.
2. **Is there really position bias?** First position (A) wins 57.9% of decided votes, but the systems are unevenly placed (Sonic is mostly in A, IndicF5 mostly in B), so we compare within each system pairing.
3. **How much do raters agree with each other?** This gives the human ceiling for this test set (how often two raters pick the same clip), needed for the headroom gate.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.home() / "projects" / "hinglish-cs-tts"
SRC = ROOT / "analysis" / "step1_4b" / "sab_meta.parquet"

# everything shown with log() is also saved to output.log, so no result lives only in kernel memory
OUTDIR = ROOT / "analysis" / "step1_4c"
OUTDIR.mkdir(parents=True, exist_ok=True)


def log(*parts):
    """Print a line and append it to output.log."""
    line = " ".join([str(part) for part in parts])
    print(line)
    with open(OUTDIR / "output.log", "a") as log_file:
        log_file.write(line + "\n")


def save_json(name, data):
    """Save data as JSON under OUTDIR, atomically (temp file, then rename)."""
    tmp = OUTDIR / (name + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str))
    tmp.replace(OUTDIR / name)


log("python", sys.version.split()[0], "| pandas", pd.__version__)
df = pd.read_parquet(SRC)
log("loaded", len(df), "rows from", SRC.name, "| outputs saved under", OUTDIR)

In [ ]:
# Vote class and the mean of the six axis ratings given to clip A and to clip B by the same rater
def vote_class(preference, model_a, model_b):
    """tie, A wins, B wins, or both named (a value such as "Gemini 2.5 Pro TTS, Sonic 3")."""
    if preference == "Tie / No Preference":
        return "tie"
    if preference == model_a:
        return "A wins"
    if preference == model_b:
        return "B wins"
    return "both named"


def mean_axes(fine_grained, key):
    """Mean of the six 1-5 axis ratings of one clip, or NaN when they are missing."""
    ratings = None
    if isinstance(fine_grained, dict):
        ratings = fine_grained.get(key)
    if not ratings:
        return np.nan
    values = [float(v) for v in ratings.values()]
    return float(np.mean(values))


classes = []
for preference, model_a, model_b in zip(df["preference_model"], df["model_a"], df["model_b"]):
    classes.append(vote_class(preference, model_a, model_b))
df["cls"] = classes
ratings_a = []
ratings_b = []
for fine_grained in df["fine_grained_eval"]:
    ratings_a.append(mean_axes(fine_grained, "detailed_ratings_a"))
    ratings_b.append(mean_axes(fine_grained, "detailed_ratings_b"))
df["ra"] = ratings_a
df["rb"] = ratings_b
log(df["cls"].value_counts().to_string())

## Question 1: what does "both named" mean?
If it means *both good*, both clips should have high axis ratings (4 to 5). If it means *both bad*, both low (1 to 2). Ties and single winners are shown for comparison.

In [ ]:
g = df.groupby("cls").agg(votes=("cls", "size"), mean_rating_A=("ra", "mean"), mean_rating_B=("rb", "mean"))
both_high = (df["ra"] >= 4) & (df["rb"] >= 4)
both_low = (df["ra"] <= 2) & (df["rb"] <= 2)
g["share_both_high"] = both_high.groupby(df["cls"]).mean()
g["share_both_low"] = both_low.groupby(df["cls"]).mean()
log(g.round(3).to_string())
both = df[df["cls"] == "both named"]
log("\nrows where both ratings are present among 'both named':", int((both["ra"].notna() & both["rb"].notna()).sum()), "of", len(both))
log("'both named': mean of (rating A + rating B)/2 =", round(float(((both["ra"] + both["rb"]) / 2).mean()), 3))

# Do the axis ratings agree with the stated winner? (share of rows where the winner's mean rating is higher / equal / lower)
dec = df[df["cls"].isin(["A wins", "B wins"])].copy()
dec["w"] = np.where(dec["cls"] == "A wins", dec["ra"] - dec["rb"], dec["rb"] - dec["ra"])
d2 = dec.dropna(subset=["w"])
log("\nwinner has a higher mean rating:", round(float((d2["w"] > 0).mean()), 3), "| equal:", round(float((d2["w"] == 0).mean()), 3),
    "| lower:", round(float((d2["w"] < 0).mean()), 3), "| rows:", len(d2))
log("share of ALL rows where every axis is 5 for both clips:", round(float(((df["ra"] == 5) & (df["rb"] == 5)).mean()), 3),
    "| every axis is 1 for both:", round(float(((df["ra"] == 1) & (df["rb"] == 1)).mean()), 3))

## Question 2: position bias, controlled for the system pairing
For each ordered pairing (system in position A vs system in position B) we look at decided votes only, and compare how often the first position wins. If there is no position bias, the win rate of a system should not depend on whether it was played first or second.

In [ ]:
dec = df[df["cls"].isin(["A wins", "B wins"])].copy()
dec["first_wins"] = (dec["cls"] == "A wins")
t = dec.groupby(["model_a", "model_b"]).agg(decided_votes=("first_wins", "size"), first_wins=("first_wins", "mean")).round(3)
log("share of decided votes where the first-position system wins, per ordered pairing:")
log(t.to_string())

systems = sorted(set(dec["model_a"]) | set(dec["model_b"]))
rows = []
for i in range(len(systems)):
    for j in range(i + 1, len(systems)):
        x = systems[i]
        y = systems[j]
        x_first = dec[(dec["model_a"] == x) & (dec["model_b"] == y)]    # x in position A
        x_second = dec[(dec["model_a"] == y) & (dec["model_b"] == x)]   # x in position B
        if len(x_first) and len(x_second):
            rows.append({
                "pair": f"{x} vs {y}",
                "x_win_rate_when_first": round(float(x_first["first_wins"].mean()), 3),
                "n_first": len(x_first),
                "x_win_rate_when_second": round(float((~x_second["first_wins"]).mean()), 3),
                "n_second": len(x_second),
            })
sw = pd.DataFrame(rows)
log("\nwin rate of the SAME system when played first versus second (no position bias would give equal numbers):")
log(sw.to_string(index=False))
log("\nweighted mean first-position win rate over decided votes:", round(float(dec["first_wins"].mean()), 3))

## Question 3: how much do raters agree (the human ceiling for this set)?
For every item (same sentence, same system A and B) with at least 2 votes, we look at all pairs of raters and count how often two raters give the same answer. We report it for the strict label (including ties and "both named") and for decided votes only.

In [ ]:
dfk = df.copy()
dfk["item"] = dfk["academic_prompt_id"].astype(str) + "|" + dfk["model_a"] + "|" + dfk["model_b"]


def agreement(frame, label_col):
    """Share of rater pairs on the same item that give the same answer, and the number of rater pairs."""
    agree = 0
    total = 0
    for _, group in frame.groupby("item"):
        labels = list(group[label_col])
        for i in range(len(labels)):
            for j in range(i + 1, len(labels)):
                total += 1
                if labels[i] == labels[j]:
                    agree += 1
    if total == 0:
        return None, total
    return agree / total, total


def is_mixed(sentence):
    """True if the sentence has both a Devanagari character and an English letter."""
    sentence = sentence or ""
    has_devanagari = False
    has_latin = False
    for ch in sentence:
        if "ऀ" <= ch <= "ॿ":
            has_devanagari = True
        if ch.isascii() and ch.isalpha():
            has_latin = True
    return has_devanagari and has_latin


strict, n_strict = agreement(dfk, "cls")
decided = dfk[dfk["cls"].isin(["A wins", "B wins"])]
dec_only, n_dec = agreement(decided, "cls")
mixed = dfk[dfk["sentence"].map(is_mixed)]
mx_strict, n_mx = agreement(mixed, "cls")
log(f"items with at least 2 votes: {int((dfk.groupby('item').size() >= 2).sum())} of {dfk['item'].nunique()}")
log(f"strict agreement (A / B / both / tie must match), {n_strict} rater pairs: {strict:.3f}")
log(f"agreement among decided votes only (A wins vs B wins), {n_dec} rater pairs: {dec_only:.3f}")
log(f"strict agreement on mixed Hindi-English sentences, {n_mx} rater pairs: {mx_strict:.3f}")
log("reference: human ceiling quoted in the literature is about 76% on clean modern TTS pairs")

## Summary (saved to `analysis/step1_4c/summary.json`)

In [ ]:
summary = {
    "vote_classes": df["cls"].value_counts().to_dict(),
    "axis_ratings_by_class": g.round(3).reset_index().to_dict("records"),
    "winner_rating_higher_share": round(float((d2["w"] > 0).mean()), 3),
    "winner_rating_equal_share": round(float((d2["w"] == 0).mean()), 3),
    "winner_rating_lower_share": round(float((d2["w"] < 0).mean()), 3),
    "all_five_both_share": round(float(((df["ra"] == 5) & (df["rb"] == 5)).mean()), 3),
    "all_one_both_share": round(float(((df["ra"] == 1) & (df["rb"] == 1)).mean()), 3),
    "first_position_win_rate_overall": round(float(dec["first_wins"].mean()), 3),
    "ordered_pairings": t.reset_index().to_dict("records"),
    "same_system_first_vs_second": sw.to_dict("records"),
    "agreement_strict": strict, "agreement_strict_rater_pairs": n_strict,
    "agreement_decided_only": dec_only, "agreement_decided_rater_pairs": n_dec,
    "agreement_strict_mixed": mx_strict, "agreement_strict_mixed_rater_pairs": n_mx,
}
save_json("summary.json", summary)
log("summary saved to", OUTDIR / "summary.json")
short_summary = {}
for key, value in summary.items():
    if not isinstance(value, list):
        short_summary[key] = value
log(json.dumps(short_summary, indent=1, default=str)[:2500])